# 03 · Seed replication

**Grand Challenge Labs · Coupling-Phase Spectroscopy**

Apply one CPS measurement contract across independent PolyPythia seeds to separate repeatable structure from initialization and data-order variation.


## Release contract

| Contract | Declared value |
|---|---|
| **Scientific question** | Which CPS signatures persist across independently trained seeds? |
| **Default path** | Run two seeds as a Colab-scale demonstration; expand the panel before making a variance claim. |
| **Evidence boundary** | Two seeds validate the replication harness. They do not estimate the population distribution. |
| **Primary outputs** | One evidence packet per seed, a manifest-level comparison, and export archive. |


## Interpretation checklist

- [ ] Verify that checkpoint, basis, data, JVP, and sweep contracts are identical across seeds.
- [ ] Interpret seed differences as variance evidence rather than automatic instrument failure.
- [ ] Reserve distributional language for a materially broader seed panel.


## Design principle

Every seed is measured with the same checkpoint revision, parameter-selection contract, batch construction, projection rank, and phase grid. The seed identifier changes; the instrument does not.

Set `CPS_SEEDS` and `CPS_REVISION` through the Colab environment. The default two-seed run is a harness check, not a population-level conclusion.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import apply_release_theme, stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
apply_release_theme()
runtime = show_environment()

In [ ]:
import dataclasses, os
from cps.notebook import show_config
from cps.pythia.config import load_probe_config

base = load_probe_config("subjects/pythia/configs/polypythia_70m_seed_study.yaml")
seeds = [int(item) for item in os.environ.get("CPS_SEEDS", "1,2").split(",") if item.strip()]
revision = os.environ.get("CPS_REVISION", "step1000")
show_config(base)
print(f"[SEEDS] revision={revision}; seeds={seeds}", flush=True)

## Execute identical probes across seeds

The runner log announces each model repository and writes an independent evidence root. This is important for later grouped cross-validation: no seed should share a manifest or reduced operator with another seed.

In [ ]:
from cps.pythia.runner import run_probe

outputs = []
for index, seed in enumerate(seeds, start=1):
    print(f"\n[SEEDS] ===== seed {seed} ({index}/{len(seeds)}) =====", flush=True)
    model = dataclasses.replace(base.model, run=f"polypythia-70m-seed{seed}", revision=revision)
    output_config = dataclasses.replace(base.output, run_name=f"polypythia-seed-{seed}")
    current = dataclasses.replace(base, model=model, output=output_config)
    outputs.append(run_probe(current))
print("[SEEDS] completed roots:", *outputs, sep="\n  - ", flush=True)

## Compare run-level observables

This table is descriptive. A serious seed study should estimate variance components and test held-out seed prediction rather than compare only maxima.

The stable run identifier is stored in `manifest.run_spec.key`. The reader below also accepts older packets that used `name`, and it reports empty coupling sets as `NaN` rather than failing.

In [ ]:
import json, pathlib, pandas as pd
from IPython.display import display

def run_identity(manifest):
    """Resolve current schema-v2 and legacy run identifiers."""
    run_spec = manifest.get("run_spec") or {}
    model_config = (manifest.get("config") or {}).get("model") or {}
    return {
        "key": (
            run_spec.get("key")
            or run_spec.get("name")
            or model_config.get("run")
            or manifest.get("model_id")
            or "unknown"
        ),
        "family": run_spec.get("family", "unknown"),
        "seed": run_spec.get("seed"),
    }

def metric_max(records, metric):
    return max(
        (record["metrics"][metric] for record in records),
        default=float("nan"),
    )

rows = []
for output in outputs:
    root = pathlib.Path(output)
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    records = json.loads((root / "couplings.json").read_text(encoding="utf-8"))
    identity = run_identity(manifest)
    if not records:
        print(f"[SEEDS] warning: no coupling records in {root}", flush=True)
    rows.append({
        "run": identity["key"],
        "seed": identity["seed"],
        "family": identity["family"],
        "revision": manifest.get("revision", "unknown"),
        "JVP backend": manifest.get("jacobian", {}).get("effective_backend", "unknown"),
        "closure max": manifest.get("projection", {}).get("maximum_closure_residual", float("nan")),
        "phase radius max": metric_max(records, "spectral_radius_max"),
        "transient gain max": metric_max(records, "finite_horizon_gain"),
    })

seed_frame = pd.DataFrame(rows)
display(seed_frame)
print("[SEEDS] Run identity resolved from the evidence manifest without modifying it.", flush=True)
print("[SEEDS] Do not interpret two points as a variance estimate.", flush=True)

## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
